In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2026, 1, 7, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 1, 7, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 66.37it/s]


In [3]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions, straddle_pricer_from_row
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
# sdf.head(50)

Classifying Trades: 100%|██████████| 530/530 [00:00<00:00, 1666.96trade/s]


In [4]:
sdf

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
0,CORR-TRAD,1646268999000000201,2026-01-07 09:22:08+00:00,2026-01-07,2029-01-08,SWAPTION_CHOOSER,USD-SOFR-COMPOUND 1D CONSTANT 3Yx5Y CHOOSER EU...,100000000000000000000.0,,False,...,NA/Swap OIS USD,QZKDXXXPW3X3,BILT,N,True,"2,477,500","2,477,500",NaN,None,None
1,CORR-TRAD,1646269000000000301,2026-01-07 09:22:11+00:00,2026-01-07,2029-01-08,SWAPTION_CHOOSER,USD-SOFR-COMPOUND 1D CONSTANT 3Yx5Y CHOOSER EU...,100000000000000000000.0,,False,...,NA/Swap OIS USD,QZKDXXXPW3X3,BILT,N,True,"2,477,500","2,477,500",NaN,None,None
2,NEWT-TRAD,1647218285000000301,2026-01-07 09:28:14+00:00,2026-01-07,2026-07-07,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 6Mx5Y PAYER EURO...,100000000000000000000.0,,False,...,NA/Swap OIS USD,QZBK1N7NJ0VF,BILT,N,False,,"819,000",NaN,None,None
3,NEWT-TRAD,1647218286000000401,2026-01-07 09:28:14+00:00,2026-01-07,2026-07-07,SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 6Mx5Y RECEIVER E...,100000000000000000000.0,,False,...,NA/Swap OIS USD,QZ2GZS235CHW,BILT,N,False,,"512,000",NaN,None,None
4,NEWT-TRAD,1647218287000000501,2026-01-07 09:28:14+00:00,2026-01-07,2026-07-07,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 6Mx5Y PAYER EURO...,100000000000000000000.0,,False,...,NA/Swap OIS USD,QZBK1N7NJ0VF,BILT,N,False,,"78,000",NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,NEWT-TRAD,1655654572000000101 / 1655654573000000201,2026-01-07 21:35:55+00:00,2026-01-07,2041-01-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 15Yx10Y PAYE...,25000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,ISWV,N,True,NaN,"4,530,000",1.0,platform=ISWV; time_delta_max=0.0s; premium_mo...,2
409,NEWT-TRAD,1655663697000000101 / 1655663698000000201,2026-01-07 21:37:18+00:00,2026-01-07,2046-01-08,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 20Yx10Y PAYE...,27000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,ISWV,N,True,"5,535,000","5,535,000",1.0,platform=ISWV; time_delta_max=0.0s; premium_mo...,2
410,NEWT-TRAD,1655795161000000101 / 1655795162000000201,2026-01-07 21:51:19+00:00,2026-01-07,2029-01-08,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y PAYER...,100000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,ISWV,N,True,"8,740,000","8,740,000",1.0,platform=ISWV; time_delta_max=0.0s; premium_mo...,2
411,NEWT-TRAD,1655812025000000101 / 1655812026000000201,2026-01-07 21:52:10+00:00,2026-01-07,2026-10-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 9Mx10Y PAYER...,100000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,ISWV,N,True,"4,200,000","4,200,000",1.0,platform=ISWV; time_delta_max=0.0s; premium_mo...,2


In [17]:
# sdf[sdf["package_type"] == "RISK_REVERSAL"]

sdf["trade_label"].value_counts()
# sdf["package_type"].value_counts()
# sdf[(sdf["package_type"] == "STRADDLE") & ((sdf["forward_label"] == "3M")) & ((sdf["tenor_label"] == "10Y"))]

trade_label
USD-SOFR-OIS Compound 1D CONSTANT 6Mx30Y RECEIVER EURO VANILLA PHYS                                                                                                                                                                                                                11
USD-SOFR-OIS Compound 1Y CONSTANT 4Mx10Y PAYER EURO VANILLA PHYS                                                                                                                                                                                                                    9
USD-SOFR-OIS Compound 1D CONSTANT 3Mx10Y RECEIVER EURO VANILLA PHYS                                                                                                                                                                                                                 7
USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEIVER EURO VANILLA PHYS                    

In [17]:
sdf[sdf["trade_label"].str.contains("1Yx10Y")]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
74,NEWT-TRAD,1654744104000000201,2026-01-07 15:51:24+00:00,2026-01-07,2027-01-07,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 1Yx10Y PAYER EUR...,250000000.0,USD,True,...,NA/Swap Fxd Flt USD,QZXSN072GFF3,BILT,N,False,,"6,029,142.55",NaN,None,None
95,NEWT-TRAD,1652865019000000801,2026-01-07 16:24:09+00:00,2026-01-07,2027-01-07,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,250000000.0,USD,True,...,NA/Swap OIS USD,QZNLQ8T0N0SX,BILT,N,True,9.9999999999,"5,954,142.85",NaN,None,None
244,NEWT-TRAD,1651977319000000201 / 1651977320000000301,2026-01-07 14:12:54+00:00,2026-01-07,2027-01-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,100000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZWXKVHB5F8V / QZMMWR8JKZQ8,BGCD,N,True,"4,940,000","4,940,000",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2
245,MODI-TRAD,1652144074000000801 / 1652231341000000501,2026-01-07 14:22:54+00:00,2026-01-07,2027-01-07,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,100000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZMMWR8JKZQ8 / QZWXKVHB5F8V,BGCD,N,True,"4,940,000","4,940,000",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2
248,NEWT-TRAD / MODI-TRAD,1652119793000000401 / 1652119794000000501,2026-01-07 14:28:18+00:00,2026-01-07,2027-01-07,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,25000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZMMWR8JKZQ8 / QZWXKVHB5F8V,BGCD,N,True,"1,242,500","1,242,500",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2
250,NEWT-TRAD / MODI-TRAD,1652200981000000101 / 1652278039000000101,2026-01-07 14:36:55+00:00,2026-01-07,2027-01-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,25000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZWXKVHB5F8V / QZMMWR8JKZQ8,BGCD,N,True,"1,242,500","1,242,500",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2
301,NEWT-TRAD / MODI-TRAD,1654088674000000101 / 1654220445000000201,2026-01-07 17:54:05+00:00,2026-01-07,2027-01-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,25000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,TPSE,N,True,"1,230,000","1,230,000",1.0,platform=TPSE; time_delta_max=0.0s; premium_mo...,2
304,NEWT-TRAD / MODI-TRAD,1654263330000000201 / 1654525957000000301,2026-01-07 18:06:15+00:00,2026-01-07,2027-01-07,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y RECEI...,130000000.0,USD,False,...,NA/Swap OIS USD,QZNLQ8T0N0SX / QZZGWPNBF5R3,TPSE,N,True,"6,137,500","6,383,000",1.0,platform=TPSE; time_delta_max=0.0s; premium_mo...,2
305,MODI-TRAD,1654348466000000301 / 1654787017000000301,2026-01-07 18:13:29+00:00,2026-01-07,2027-01-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,50000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZWXKVHB5F8V / QZMMWR8JKZQ8,BGCD,N,True,"2,455,000","2,455,000",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2
306,NEWT-TRAD / MODI-TRAD,1654348467000000401 / 1654756786000000701,2026-01-07 18:13:41+00:00,2026-01-07,2027-01-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,50000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZWXKVHB5F8V / QZMMWR8JKZQ8,BGCD,N,True,"2,455,000","2,455,000",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2


In [15]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))
pricer

QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x00000276EC4C7060> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x00000276ECCDC690> >, _meta_data={'timestamp': datetime.datetime(2026, 1, 7, 0, 0)})

In [20]:
# sdf.loc[285], sdf.loc[286]
# .iloc[0].to_dict()
# df[df["Original Dissemination Identifier"] == 1653435994000001301]
qls, bpvol = straddle_pricer_from_row(sdf.loc[244], pricer)

In [28]:
new_ql_swaption_pricing_engine = ql.BachelierSwaptionEngine(pricer.handle(), ql.QuoteHandle(ql.SimpleQuote(bpvol / 10_000)), pricer.daycounter())
qls.setPricingEngine(new_ql_swaption_pricing_engine)
(2 * qls.vega()) / 10_000

64828.88651203758

In [9]:
ids = [
    # "1656268005000000101",
	# "1655894585000000101",
	# "1655795163000000301",
	# "1655795162000000201",
	# "1656192606000000601",
	# "1655795161000000101",
    "1655445236000000201"
]

failed_trade_entity_ids = [
    "1650906615000000601",
    "1650906617000000801",
    "1651137506000001401",
    "1651151962000000301",
    "1651505182000001001", "1651391705000000401",
    "1651402069000000901", "1651505181000000901",
    "1652241159000002101",
    "1652241160000002201",
    "1652312782000000301",
    "1652312783000000401",
    "1652332802000000601",
    "1652332803000000701",
    "1652391592000000201",
    "1652400689000001301",
    "1652411992000002201",
    "1652462266000000201",
    "1652609798000000901",
    "1652617055000001101",
    "1652620860000000101",
    "1652629192000000801",
    "1652629193000000901",
    "1652629194000001001",
    "1652629195000001101",
    "1652629196000001201",
    "1652629197000001301",
    "1652629199000001501",
    "1652629203000001901",
    "1652629204000002001",
    "1652695420000001601",
    "1652695421000001701",
    "1652741543000000201", "1652702784000000501",
    "1652758698000002101", "1654806170000000201", "1652753757000001501",
    "1652758684000000701",
    "1652979886000000101",
    "1652984676000001001",
    "1653135290000000301",
    "1653659269000000601", "1653470278000000201",
    "1653470279000000301", "1653659270000000701",
    "1656033095000000401", "1653659278000000401",
    "1654215492000001001",
    "1654220455000000301",
    "1654401073000000301",
    "1654401076000000601",
    "1654423029000000601", "1654806171000000301",
    "1654455321000000301",
    "1654466492000000401",
    "1654471547000000201", "1655071379000001901",
    "1654533333000000501", "1655071378000001801",
    "1654533334000000601", "1655071374000001401",
    "1654533337000000901", "1655071369000000901",
    "1654581878000000401",
    "1654666632000000301",
    "1654782628000000201",
    "1654782629000000301",
    "1654783393000000101",
    "1654783394000000201",
    "1654783619000000801",
    "1654783620000000901",
    "1654783621000001001",
    "1654783622000001101",
    "1654783998000000101",
    "1654784487000000301",
    "1654784488000000401",
    "1654808641000000101",
    "1654808642000000201",
    "1654821412000000501",
    "1654870624000000301",
    "1654902757000000301",
    "1654902758000000401",
    "1654903104000000101",
    "1654903105000000201",
    "1654973112000000301",
    "1655071372000001201", "1655973979000000301",
    "1655071373000001301", "1655965958000000201",
    "1655071376000001601", "1655973977000000101",
    "1655071377000001701", "1655973978000000201",
    "1655445236000000201",
    "1655445237000000301",
    "1655925196000000701",
    "1655973980000000401",
    "1655973982000000601",
    "1655973983000000701",
    "1655973984000000801",
    "1656033094000000301",
    "1656162868000000101",
]


# df[df["Dissemination Identifier"].isin(failed_trade_entity_ids)].to_dict(orient="records")
df[df["Dissemination Identifier"].isin(failed_trade_entity_ids)]["Event type"].value_counts()
df[df["Dissemination Identifier"].isin(failed_trade_entity_ids)]["Action type"].value_counts()

# df["Action type"].value_counts()
# df["Event type"].value_counts()

Action type
NEWT    55
TERM    32
CORR    12
MODI     3
Name: count, dtype: int64